# Camada Silver — Tratamento dos Indicadores Econômicos

Este notebook lê os arquivos brutos (bronze) do S3, trata os tipos de dado
(data e valor, que chegam como texto da API) e remove duplicatas, salvando
o resultado tratado de volta no S3, em formato Parquet, na camada silver.

## Autenticação
Cria os campos de entrada (widgets) para colar as credenciais da AWS
manualmente a cada execução, evitando expor a chave no código-fonte.

In [0]:
# Cria os campos de texto no topo do notebook para inserir as credenciais da AWS
dbutils.widgets.text("access_key", "")
dbutils.widgets.text("secret_key", "")

In [0]:
# Recupera os valores digitados nos widgets e conecta ao S3
access_key = dbutils.widgets.get("access_key")
secret_key = dbutils.widgets.get("secret_key")

import boto3

s3 = boto3.client(
    "s3",
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name="sa-east-1"
)

## Leitura da camada Bronze
Busca os 3 arquivos brutos (SELIC, IPCA, Dólar) salvos na pasta do dia atual
e os carrega em memória, ainda no formato original (JSON cru da API).

In [0]:
import json
from datetime import datetime

# Define a data de hoje, usada para localizar a pasta correta no bronze
data_hoje = datetime.today().strftime("%Y-%m-%d")
nomes = ["selic", "ipca", "dolar"]

dados_bronze = {}

# Para cada indicador, busca o arquivo correspondente no S3 e converte de JSON para dicionário Python
for nome in nomes:
    caminho = f"bronze/{data_hoje}/{nome}.json"
    resposta = s3.get_object(Bucket="pipeline-economico-hugoqueiroz", Key=caminho)
    conteudo = resposta["Body"].read()
    dados_bronze[nome] = json.loads(conteudo)
    print(f"{nome}: {len(dados_bronze[nome])} registros lidos")

selic: 679 registros lidos
ipca: 32 registros lidos
dolar: 679 registros lidos


## Tratamento dos dados (pandas)
Converte cada série para uma tabela (DataFrame), corrigindo os tipos:
- `data`: texto → data (datetime)
- `valor`: texto → número decimal (float)

Também remove eventuais linhas duplicadas.

Optei por pandas em vez de PySpark porque o volume de dados (~1.400 linhas
no total) é pequeno o suficiente para processar em memória local, sem a
necessidade de processamento distribuído.

In [0]:
import pandas as pd

dados_silver = {}

# Para cada indicador, transforma em DataFrame e corrige os tipos de dado
for nome, registros in dados_bronze.items():
    df = pd.DataFrame(registros)
    df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
    df["valor"] = df["valor"].astype(float)
    df = df.drop_duplicates()
    dados_silver[nome] = df
    print(f"{nome}: {df.shape[0]} linhas, {df.shape[1]} colunas")
    print(df.dtypes)
    print()

selic: 679 linhas, 2 colunas
data     datetime64[ns]
valor           float64
dtype: object

ipca: 32 linhas, 2 colunas
data     datetime64[ns]
valor           float64
dtype: object

dolar: 679 linhas, 2 colunas
data     datetime64[ns]
valor           float64
dtype: object



## Salvamento da camada Silver
Salva cada indicador tratado no S3, em formato Parquet — que, diferente do
CSV, preserva os tipos de dado (data e número) ao ser lido novamente.

In [0]:
import io

# Para cada indicador tratado, converte para Parquet e salva no S3, particionado por data
for nome, df in dados_silver.items():
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False)
    caminho = f"silver/{data_hoje}/{nome}.parquet"
    s3.put_object(
        Bucket="pipeline-economico-hugoqueiroz",
        Key=caminho,
        Body=buffer.getvalue()
    )
    print(f"Salvo: {caminho}")

Salvo: silver/2026-09-14/selic.parquet
Salvo: silver/2026-09-14/ipca.parquet
Salvo: silver/2026-09-14/dolar.parquet
